# Sefaria manuscript facsimiles → local mirror

Downloads Sefaria's manuscript images that are mapped to a Hebrew text and builds a
**verse → facsimile-image** index you can browse offline and later wire into your
interlinear pages (the 📖 link, same as your Glück facsimiles).

Data comes from Sefaria's public API (cleaner than scraping the SPA HTML):

| endpoint | gives |
|---|---|
| `api/shape/<Book>` | verse count per chapter (to enumerate refs) |
| `api/manuscripts/<Book.ch>` | image entries overlapping that chapter, each with `anchorRefExpanded` = exact verses on that folio |
| `api/v3/texts/<Book.ch>` | Hebrew + English verse text (optional, for the local viewer) |
| `manuscripts.sefaria.org/...jpg` | the facsimile image itself (full + thumbnail) |

**One folio image covers many verses** (e.g. `Psalms 40:7–42:7`), so images are
**downloaded once and deduplicated by URL**; the verse→image map just references them.

### Outputs (under `OUTDIR`)
```
images/<slug>/<file>.jpg      full folios, deduped
thumbs/<slug>/<file>.jpg      thumbnails, deduped
cache/manuscripts/<Book>/<ch>.json   raw API cache (resumable)
cache/text/<Book>/<ch>.json          text cache
image_manifest.json           url → {local_path, sha256, bytes}
verse_image_map.json          "Psalms 42:4" → [ {manuscript, page_id, anchorRef, image, sefaria_url, ...} ]   ← the file you'll consume
html/index.html + html/<Book>/<ch>.html   offline viewer with 📖 links
```
The notebook is **resumable**: per-chapter API responses and downloaded images are cached, so re-running skips finished work. Run it incrementally to grow from one book to the whole Tanakh.

## 0 · Config

In [ ]:
from pathlib import Path

# --- what to fetch -----------------------------------------------------------
BOOKS = ["Psalms"]          # start small; add more or use FULL_TANAKH below
# FULL_TANAKH (uncomment to mirror everything Leningrad covers):
TANAKH = [
    "Genesis","Exodus","Leviticus","Numbers","Deuteronomy",
    "Joshua","Judges","I Samuel","II Samuel","I Kings","II Kings",
    "Isaiah","Jeremiah","Ezekiel",
    "Hosea","Joel","Amos","Obadiah","Jonah","Micah","Nahum","Habakkuk",
    "Zephaniah","Haggai","Zechariah","Malachi",
    "Psalms","Proverbs","Job","Song of Songs","Ruth","Lamentations",
    "Ecclesiastes","Esther","Daniel","Ezra","Nehemiah",
    "I Chronicles","II Chronicles",
]
BOOKS = TANAKH

OUTDIR = Path("sefaria_manuscripts")

# Restrict to specific manuscripts by slug, or None for all that Sefaria returns.
# e.g. {"leningrad-codex-(1008-ce)"}  (Leningrad covers the whole Tanakh)
MANUSCRIPT_WHITELIST = None

# --- images ------------------------------------------------------------------
DOWNLOAD_FULL   = True      # full-res folios (~3-4 MB each)
DOWNLOAD_THUMBS = True      # thumbnails (~15 KB) used for the viewer gallery

# --- text (for the offline viewer) ------------------------------------------
FETCH_TEXT    = True
TEXT_VERSIONS = ["hebrew", "english"]   # Sefaria default versions for each

# --- politeness / robustness -------------------------------------------------
REQUEST_DELAY = 0.3         # seconds between API calls
MAX_RETRIES   = 4
TIMEOUT       = 60
UA = "manuscript-mirror/1.0 (personal research; bible alignment project)"

SEFARIA = "https://www.sefaria.org"

## 1 · HTTP helpers

In [ ]:
import time, json, hashlib, requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

OUTDIR.mkdir(parents=True, exist_ok=True)
(OUTDIR / "cache" / "manuscripts").mkdir(parents=True, exist_ok=True)
(OUTDIR / "cache" / "text").mkdir(parents=True, exist_ok=True)

sess = requests.Session()
sess.headers["User-Agent"] = UA
_retry = Retry(total=MAX_RETRIES, backoff_factor=1.0,
               status_forcelist=[429, 500, 502, 503, 504],
               allowed_methods=["GET"])
sess.mount("https://", HTTPAdapter(max_retries=_retry))

def api_get(path, **params):
    r = sess.get(f"{SEFARIA}/{path}", params=params, timeout=TIMEOUT)
    r.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return r.json()

def get_shape(book):
    d = api_get(f"api/shape/{book.replace(' ', '_')}")
    d = d[0] if isinstance(d, list) else d
    return d["chapters"]              # list[int]: verses per chapter

def ref_to_url(ref):
    '''Psalms 42:4 / Song of Songs 1:1 -> Psalms.42.4 / Song_of_Songs.1.1'''
    book, cv = ref.rsplit(" ", 1)
    return f"{book.replace(' ', '_')}.{cv.replace(':', '.')}"

def sefaria_view_url(ref):
    return f"{SEFARIA}/{ref_to_url(ref)}?lang=bi&with=manuscripts&lang2=en"

print("session ready ->", OUTDIR.resolve())

## 2 · Crawl manuscript metadata → `verse_image_map`

Queries **per chapter** (≈30× fewer requests than per verse). Each returned entry's
`anchorRefExpanded` lists the exact verses of that chapter present on the folio, so the
verse→image map is precise. Responses are cached to `cache/manuscripts/` for resume.

In [ ]:
from collections import defaultdict

verse_image_map = defaultdict(list)   # "Psalms 42:4" -> [record, ...]
images = {}                           # image_url -> {slug, basename, thumb_url}

def _basename(url):
    return url.rstrip("/").split("/")[-1]

def crawl_chapter(book, ch):
    cache = OUTDIR / "cache" / "manuscripts" / book.replace(" ", "_") / f"{ch}.json"
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        data = json.loads(cache.read_text("utf-8"))
    else:
        data = api_get(f"api/manuscripts/{book.replace(' ', '_')}.{ch}")
        cache.write_text(json.dumps(data, ensure_ascii=False), "utf-8")
    return data

def ingest(entry):
    slug = entry["manuscript_slug"]
    if MANUSCRIPT_WHITELIST and slug not in MANUSCRIPT_WHITELIST:
        return
    img = entry["image_url"]
    thumb = entry.get("thumbnail_url")
    images.setdefault(img, {"slug": slug, "basename": _basename(img),
                            "thumb_url": thumb,
                            "thumb_basename": _basename(thumb) if thumb else None})
    rec = {
        "manuscript": entry["manuscript"]["title"],
        "slug": slug,
        "page_id": entry.get("page_id"),
        "anchorRef": entry.get("anchorRef"),
        "image_url": img,
        "thumb_url": thumb,
    }
    for vref in entry.get("anchorRefExpanded", []):
        # avoid duplicate (same image) per verse
        if not any(r["image_url"] == img for r in verse_image_map[vref]):
            verse_image_map[vref].append(dict(rec))

for book in BOOKS:
    chapters = get_shape(book)
    print(f"{book}: {len(chapters)} chapters", end="  ")
    hits = 0
    for ch in range(1, len(chapters) + 1):
        for entry in crawl_chapter(book, ch):
            ingest(entry); hits += 1
    print(f"-> {hits} image-entries")

print(f"\nverses with images : {len(verse_image_map)}")
print(f"unique images      : {len(images)}")

## 3 · Download images (deduplicated)

In [ ]:
MANIFEST = OUTDIR / "image_manifest.json"
manifest = json.loads(MANIFEST.read_text("utf-8")) if MANIFEST.exists() else {}

def fetch_file(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return dest.stat().st_size, None
    with sess.get(url, stream=True, timeout=TIMEOUT) as r:
        r.raise_for_status()
        h = hashlib.sha256(); n = 0
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 16):
                f.write(chunk); h.update(chunk); n += len(chunk)
        tmp.rename(dest)
    time.sleep(REQUEST_DELAY)
    return n, h.hexdigest()

total = len(images)
for i, (url, meta) in enumerate(images.items(), 1):
    rel = f"images/{meta['slug']}/{meta['basename']}"
    if DOWNLOAD_FULL:
        size, sha = fetch_file(url, OUTDIR / rel)
        m = manifest.get(url, {})
        m.update({"local_path": rel, "bytes": size})
        if sha: m["sha256"] = sha
        manifest[url] = m
    if DOWNLOAD_THUMBS and meta.get("thumb_url"):
        trel = f"thumbs/{meta['slug']}/{meta['thumb_basename']}"
        fetch_file(meta["thumb_url"], OUTDIR / trel)
        manifest.setdefault(url, {})["thumb_path"] = trel
    if i % 25 == 0 or i == total:
        MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=1), "utf-8")
        print(f"  {i}/{total}")

MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=1), "utf-8")
mb = sum(v.get("bytes", 0) for v in manifest.values()) / 1e6
print(f"done: {len(manifest)} images, {mb:.1f} MB on disk")

## 4 · Fetch verse text (optional, for the viewer)

In [ ]:
texts = {}   # "Psalms 42:4" -> {"he": "...", "en": "..."}

def fetch_text_chapter(book, ch):
    cache = OUTDIR / "cache" / "text" / book.replace(" ", "_") / f"{ch}.json"
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        return json.loads(cache.read_text("utf-8"))
    params = "&".join(f"version={v}" for v in TEXT_VERSIONS)
    d = api_get(f"api/v3/texts/{book.replace(' ', '_')}.{ch}?{params}")
    out = {}
    for v in d.get("versions", []):
        lang = v.get("language")
        t = v.get("text")
        out[lang] = t if isinstance(t, list) else [t]
    cache.write_text(json.dumps(out, ensure_ascii=False), "utf-8")
    return out

if FETCH_TEXT:
    for book in BOOKS:
        chapters = get_shape(book)
        for ch in range(1, len(chapters) + 1):
            ct = fetch_text_chapter(book, ch)
            he = ct.get("he", []); en = ct.get("en", [])
            for i in range(max(len(he), len(en))):
                ref = f"{book} {ch}:{i+1}"
                texts[ref] = {
                    "he": he[i] if i < len(he) else "",
                    "en": en[i] if i < len(en) else "",
                }
        print(f"text: {book} ok")
    print(f"verses with text: {len(texts)}")
else:
    print("FETCH_TEXT = False (skipped)")

## 5 · Export `verse_image_map.json` (the file you'll wire into your project)

In [ ]:
# Resolve each record's image_url -> local path, add the Sefaria manuscript-view URL.
export = {}
for ref in sorted(verse_image_map, key=lambda r: (r.rsplit(" ",1)[0],
                  *[int(x) for x in r.rsplit(" ",1)[1].split(":")])):
    recs = []
    for r in verse_image_map[ref]:
        m = manifest.get(r["image_url"], {})
        recs.append({
            "manuscript": r["manuscript"],
            "slug":       r["slug"],
            "page_id":    r["page_id"],
            "anchorRef":  r["anchorRef"],
            "image":      m.get("local_path"),
            "thumb":      m.get("thumb_path"),
            "image_url":  r["image_url"],
            "sefaria_url": sefaria_view_url(ref),
        })
    export[ref] = recs

(OUTDIR / "verse_image_map.json").write_text(
    json.dumps(export, ensure_ascii=False, indent=1), "utf-8")
print("wrote verse_image_map.json :", len(export), "verses")
print("\nexample —", "Psalms 42:4")
print(json.dumps(export.get("Psalms 42:4", "n/a"), ensure_ascii=False, indent=1))

## 6 · Build the offline viewer

`html/index.html` → per-book chapter lists → per-chapter pages.
Each verse shows Hebrew (RTL) + English and a 📖 link per folio that opens the
**local** facsimile in a new tab, anchored to the page that contains that verse.
This is a working prototype of the link you want on every interlinear verse.

In [ ]:
import html as _html

HTMLDIR = OUTDIR / "html"
HTMLDIR.mkdir(exist_ok=True)

CSS = '''
body{font-family:system-ui,sans-serif;max-width:920px;margin:2rem auto;padding:0 1rem;
 line-height:1.5;color:#1a1a1a;background:#fafaf8}
a{color:#7a5;text-decoration:none} a:hover{text-decoration:underline}
.v{padding:.45rem 0;border-bottom:1px solid #eee}
.n{color:#999;font-size:.8rem;margin-inline-end:.4rem}
.he{font-size:1.5rem;direction:rtl;text-align:right;font-family:"SBL Hebrew","Times New Roman",serif}
.en{color:#444;font-size:.95rem;margin-top:.15rem}
.fx{font-size:1.15rem;margin-inline-start:.3rem;cursor:pointer}
.gallery img{height:120px;margin:4px;border:1px solid #ddd;border-radius:4px}
nav{margin:1rem 0;font-size:.9rem} h1{font-size:1.4rem}
.ms{color:#aaa;font-size:.75rem}
'''

def chapter_refs(book, ch, nverses):
    return [f"{book} {ch}:{v}" for v in range(1, nverses + 1)]

def write_chapter_page(book, ch, nverses, prev_ref, next_ref):
    bslug = book.replace(" ", "_")
    page = HTMLDIR / bslug / f"{ch}.html"
    page.parent.mkdir(parents=True, exist_ok=True)
    rel = "../.."   # html/<Book>/<ch>.html -> OUTDIR root

    # gallery: unique images covering this chapter, in folio order
    seen, gallery = set(), []
    for v in range(1, nverses + 1):
        for r in export.get(f"{book} {ch}:{v}", []):
            if r["image_url"] in seen: continue
            seen.add(r["image_url"]); gallery.append(r)

    rows = []
    for v in range(1, nverses + 1):
        ref = f"{book} {ch}:{v}"
        recs = export.get(ref, [])
        t = texts.get(ref, {})
        he = t.get("he", ""); en = t.get("en", "")
        links = ""
        for i, r in enumerate(recs):
            target = (f'{rel}/' + r["image"]) if r.get("image") else r["image_url"]
            sup = "" if len(recs) == 1 else f"<sup>{i+1}</sup>"
            tip = _html.escape(f'{r["manuscript"]} · {r["anchorRef"]}')
            links += f'<a class="fx" target="_blank" title="{tip}" href="{target}">📖{sup}</a>'
        rows.append(
            f'<div class="v" id="v{v}"><span class="n">{v}</span>'
            f'<span class="he">{he}</span>{links}'
            + (f'<div class="en">{en}</div>' if en else "")
            + "</div>")

    gal = "".join(
        f'<a target="_blank" href="{rel}/{r["image"] or ""}">'
        f'<img loading="lazy" src="{rel}/{r.get("thumb") or r.get("image") or ""}" '
        f'title="{_html.escape(r["manuscript"]+" "+(r["page_id"] or ""))}"></a>'
        for r in gallery)

    nav = '<nav>'
    if prev_ref: nav += f'<a href="{prev_ref}.html">← {prev_ref}</a> &nbsp; '
    nav += f'<a href="../index.html">index</a>'
    if next_ref: nav += f' &nbsp; <a href="{next_ref}.html">{next_ref} →</a>'
    nav += '</nav>'

    page.write_text(
        f'<!doctype html><meta charset="utf-8"><title>{book} {ch}</title>'
        f'<style>{CSS}</style><h1>{book} {ch}</h1>{nav}'
        + "".join(rows)
        + (f'<h2 class="ms">folios</h2><div class="gallery">{gal}</div>' if gal else "")
        + nav, "utf-8")

# build pages + indexes
index_links = []
for book in BOOKS:
    chapters = get_shape(book)
    n = len(chapters)
    ch_links = []
    for ch in range(1, n + 1):
        prev_ref = str(ch - 1) if ch > 1 else None
        next_ref = str(ch + 1) if ch < n else None
        write_chapter_page(book, ch, chapters[ch - 1], prev_ref, next_ref)
        ch_links.append(f'<a href="{ch}.html">{ch}</a>')
    bslug = book.replace(" ", "_")
    (HTMLDIR / bslug / "index.html").write_text(
        f'<!doctype html><meta charset="utf-8"><title>{book}</title><style>{CSS}</style>'
        f'<h1>{book}</h1><nav><a href="../index.html">all books</a></nav><p>'
        + " · ".join(ch_links) + "</p>", "utf-8")
    index_links.append(f'<li><a href="{bslug}/index.html">{book}</a></li>')

(HTMLDIR / "index.html").write_text(
    f'<!doctype html><meta charset="utf-8"><title>Sefaria manuscripts</title>'
    f'<style>{CSS}</style><h1>Sefaria manuscript mirror</h1><ul>'
    + "".join(index_links) + "</ul>", "utf-8")

print("viewer ->", (HTMLDIR / "index.html").resolve())

## Wiring it into your interlinear pages

`verse_image_map.json` is keyed by Sefaria ref (`"Psalms 42:4"`). For each interlinear
verse, look up the ref and emit the same 📖 you use for Glück — point it at either the
local image or `sefaria_url`:

```python
import json
M = json.load(open("sefaria_manuscripts/verse_image_map.json", encoding="utf-8"))

def facsimile_link(ref):           # ref like "Psalms 42:4"
    recs = M.get(ref)
    if not recs: return ""
    href = recs[0]["image"] or recs[0]["image_url"]   # or recs[0]["sefaria_url"]
    return f'<a href="{href}" target="_blank" title="{recs[0]["manuscript"]}">📖</a>'
```

Ref-format note: your interlinear keys may differ (book naming, chapter:verse vs
chapter.verse). Normalize to Sefaria's `"Book ch:vs"` before the lookup. Psalms numbering
also differs from some traditions — Sefaria/Leningrad follows the Masoretic numbering.